# Profiling Regional Grid Load and Outages with PROC CHART

## Executive Summary

A grid operations analyst uses PROC CHART to profile a synthetic sample of feeder-circuit meter readings across four service regions and four generation sources. The notebook walks through vertical bar, horizontal bar, block, and pie charts to summarize the generation mix, compare mean and total load by region, expose the evening demand peak by hour, and rank outage minutes by source — the kind of fast, text-output exploration an energy-and-utilities team runs before committing to a graphical report. The DATA step requests 1,200 rows; this notebook was rendered in unlicensed mode, which caps the working dataset at the first 100 readings, so every chart below summarizes that 100-row sample.

## Data Sources

| Dataset | Rows | Description |
|---------|------|-------------|
| `grid_ops` | 100 (synthetic sample) | One row per feeder-circuit meter reading on a regional grid, generated inline with `call streaminit(20260531)` and `rand()`. The DATA step loop asks for 1,200 readings, but unlicensed mode caps the saved dataset at 100 observations, so the charts below describe those 100 rows. |

# Profiling Regional Grid Load and Outages with PROC CHART

PROC CHART produces character-based bar, block, and pie charts directly in the listing — no ODS Graphics device required. That makes it a quick first-pass exploration tool for a grid operations team that wants to *see* the shape of its load and reliability data before building polished GCHART or SGPLOT visuals.

In this notebook we:

1. Generate a synthetic month of feeder-circuit meter readings for a regional grid operator.
2. Chart the **generation mix** (share of readings by source).
3. Compare **mean and total delivered load** across service regions.
4. Expose the **evening demand peak** by hour of day.
5. Use a **block chart** to cross region against generation source.
6. Rank **outage minutes** by source to find the least reliable feed.

Every statement and option below is standard SAS 9.4 PROC CHART syntax.

## Step 1 — Generate the synthetic operations data

The DATA step below fabricates meter readings in a loop of 1,200 iterations. Each reading is assigned a service region, a generation source (Gas dominates, with Wind, Solar, and Hydro making up the rest), and an hour of day. Load is higher in the 17:00–21:00 evening window (and we flag those readings as peak), and we draw outage minutes from a Poisson distribution. `call streaminit` fixes the random seed so the data is reproducible.

The NOTE in the log shows the practical result: this run is in unlicensed mode, so the saved `grid_ops` dataset is capped at the first 100 of those readings (100 rows, 7 columns). Every chart that follows summarizes that 100-row sample, and each PROC CHART log confirms it read 100 rows.

In [1]:
/* Synthetic feeder-circuit operations for a regional grid operator */
data grid_ops;
    call streaminit(20260531);
    length region $12 source $9 peak_flag $3;
    array regions[4] $12 _temporary_
        ('North','South','East','West');
    do meter_id = 1 to 1200;
        r = ceil(4 * rand('uniform'));
        region = regions[r];
        u = rand('uniform');
        if u < 0.42 then source = 'Gas';
        else if u < 0.70 then source = 'Wind';
        else if u < 0.88 then source = 'Solar';
        else source = 'Hydro';
        hour = floor(24 * rand('uniform'));
        base = 18 + 5 * (region = 'North') + 3 * (region = 'West');
        if hour >= 17 and hour <= 21 then do;
            load_mwh = base + 12 + 6 * rand('normal');
            peak_flag = 'Yes';
        end;
        else do;
            load_mwh = base + 4 * rand('normal');
            peak_flag = 'No';
        end;
        if load_mwh < 0 then load_mwh = 0;
        outage_min = rand('poisson', 2.5);
        output;
    end;
    drop r u base;
run;

NOTE: DATA grid_ops

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote grid_ops (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.01 seconds
  cpu   0.01 seconds


## Step 2 — Generation mix

What share of readings does each generation source account for? A vertical bar chart with `TYPE=PERCENT` answers this directly: bar heights are the percentage of all observations falling in each source category. Because `source` is a character variable, PROC CHART treats its values as discrete categories automatically.

In [2]:
proc chart data=grid_ops;
    vbar source / type=percent;
run;


Percent of SOURCE

         |              *****       
         |              *****       
   40.00 +              *****       
         |              *****       
         |              *****       
   30.00 +              *****       
         |       *****  *****       
         |       *****  *****       
   20.00 +       *****  *****       
         |       *****  *****       
         |*****  *****  *****       
         |*****  *****  *****  *****
   10.00 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          Hydro  Wind    Gas   Solar
                    SOURCE


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Step 3 — Mean delivered load by region

Now we switch from counting to summarizing a measurement. Naming `load_mwh` in `SUMVAR=` with `TYPE=MEAN` makes the bar length the average load per region, and we request the statistics columns explicitly: `MEAN` prints the average beside each bar and `CFREQ` adds a cumulative-frequency column. North carries the highest average delivered load (mean about 28.0 MWh), consistent with the regional offset built into the data, while South is lowest (about 19.8 MWh).

We also pass `ASCENDING`, intending to order the bars from lowest to highest mean. Note in the output that the bars are **not** reordered — they appear in category order (West, South, East, North), with means 24.2, 19.8, 21.7, 28.0 that do not run shortest-to-longest. Reordering bars by the chart statistic is not yet wired up in this PROC CHART implementation, so `ASCENDING`/`DESCENDING` are accepted but currently have no effect; read the ranking from the printed `Mean` column instead.

In [3]:
proc chart data=grid_ops;
    hbar region / sumvar=load_mwh type=mean
                  cfreq mean ascending;
run;


Mean of REGION

REGION                                                  Mean           N     Percent
                                                                                    
  West  **********************************             24.17       24.17       23.00
 South  ****************************                   19.80       43.97       21.00
  East  *******************************                21.72       65.69       29.00
 North  ****************************************       28.03       93.72       27.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Step 4 — Total load by region

Here `TYPE=SUM` makes each bar the *total* delivered load for the region rather than the average, so the taller bars mark the regions delivering the most aggregate energy across the sampled readings. North again leads on total delivered load.

The statement also requests `SUBGROUP=peak_flag`, asking PROC CHART to split each bar by whether the reading fell in the evening peak window. In SAS this segments each bar with a distinct glyph per subgroup level and prints a legend. This implementation does not yet render subgroup segmentation (a documented capability gap), so the bars come out as solid `*` runs with no per-segment breakdown — the bar *totals* are correct, but the peak-vs-offpeak split is not shown here. To see how much load lands in peak hours, use the hour-of-day view in Step 5.

In [4]:
proc chart data=grid_ops;
    vbar region / sumvar=load_mwh type=sum
                  subgroup=peak_flag;
run;


Sum of REGION

         |                     *****
         |                     *****
         |                     *****
     600 +              *****  *****
         |*****         *****  *****
         |*****         *****  *****
         |*****         *****  *****
     400 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
     200 +*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |*****  *****  *****  *****
         |                          
         +--------------------------+
          West   South  East   North
                    REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Step 5 — Load shape across the day

`hour` is continuous, so we pin the binning ourselves with an explicit `MIDPOINTS=` list at 4-hour centers (2, 6, 10, 14, 18, 22). Each bar shows the mean delivered load for readings near that hour. The bar centered on 18 should stand out — that is the evening demand peak we built into the data.

In [5]:
proc chart data=grid_ops;
    vbar hour / sumvar=load_mwh type=mean
                midpoints=2 6 10 14 18 22;
run;


Mean of HOUR

         |                            *****       
         |                            *****       
   30.00 +                            *****       
         |                            *****  *****
         |                            *****  *****
         |              *****         *****  *****
   20.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
   10.00 +*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |*****  *****  *****  *****  *****  *****
         |                                        
         +----------------------------------------+
            2      6     10     14     18     22  
                            HOUR


NOTE: PROC CHART data=grid_ops


## Step 6 — Region by generation source (block chart)

A BLOCK chart renders a small two-way table as a grid of blocks. With `GROUP=source` and `SUMVAR=load_mwh / TYPE=MEAN`, the chart crosses each region against the generation source serving it, with block height proportional to mean load — a compact way to spot which region/source combinations carry the heaviest average load.

In [6]:
proc chart data=grid_ops;
    block region / sumvar=load_mwh type=mean
                   group=source;
run;


Block chart of Mean of REGION by SOURCE

                          /####\
  /####\                  /####\
  /####\          /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  /####\  /####\  /####\  /####\
  +----+  +----+  +----+  +----+
   West   South    East   North 
             REGION


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Step 7 — Generation mix as a pie chart

The same source-share information from Step 2, drawn as a pie. PIE with `TYPE=PERCENT` sizes each slice by its percentage of total readings and prints a legend of slice percentages alongside the figure.

In [7]:
proc chart data=grid_ops;
    pie source / type=percent;
run;


Pie chart of SOURCE

          SOURCE     Percent   Percent  Slice
----------------  ----------  --------  ------------------------------
           Hydro       14.00     14.0%  ####
            Wind       28.00     28.0%  ********
             Gas       45.00     45.0%  ++++++++++++++
           Solar       13.00     13.0%  OOOO


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Step 8 — Outage minutes by source

Finally we rank reliability. `SUMVAR=outage_min` with `TYPE=SUM` totals outage minutes per source. We pass `DESCENDING` to try to float the worst-performing source to the top, but as in Step 3 the bars are not reordered — they print in category order (Hydro, Wind, Gas, Solar) and bar reordering is not yet implemented. Read the ranking from the printed `Sum` column: Gas accounts for the most total outage minutes in this sample (122), well ahead of Wind (64), Solar (43), and Hydro (38). That tracks the generation mix — Gas serves the most readings, so it accumulates the most outage minutes overall.

In [8]:
proc chart data=grid_ops;
    hbar source / sumvar=outage_min type=sum descending;
run;


Sum of SOURCE

SOURCE                                                   Sum        Cum.     Percent
                                                                     Sum            
 Hydro  ************                                      38          38       14.00
  Wind  *********************                             64         102       28.00
   Gas  ****************************************         122         224       45.00
 Solar  **************                                    43         267       13.00


NOTE: PROC CHART data=grid_ops

NOTE: 100 rows read from dataset 'grid_ops'.


## Interpreting the results

Reading the charts together gives the operations team a fast situational picture (over the 100-row sample this run captured):

- **Generation mix (Steps 2 and 7).** Gas carries the largest share of readings (about 45%), with Wind second (about 28%) and Hydro and Solar trailing (about 14% and 13%) — the vertical bar and pie tell the same story two ways, a useful sanity check.
- **Load by region (Steps 3 and 4).** The mean-load HBAR shows North running the highest average delivered load (mean ~28 MWh) and South the lowest (~20 MWh), consistent with the regional offset built into the data. The SUM chart confirms North also delivers the most total energy.
- **Daily load shape (Step 5).** The midpoint bar centered on hour 18 is clearly the tallest, confirming the 17:00–21:00 demand peak we built into the data — exactly where a utility would focus demand-response and capacity planning.
- **Reliability (Step 8).** Totaling outage minutes by source surfaces Gas as the largest contributor of downtime in this sample (122 minutes), the natural next target for maintenance review — though that mostly reflects Gas serving the most readings.

Two display options used here — `ASCENDING`/`DESCENDING` bar reordering (Steps 3 and 8) and `SUBGROUP=` bar segmentation (Step 4) — are accepted by the parser but not yet rendered by this PROC CHART implementation, so the rankings and splits are read from the printed statistic columns rather than from the bar order or shading.

PROC CHART is character-output only, so for board-ready visuals the team would move these same views to PROC GCHART or PROC SGPLOT. But as a zero-setup first pass over a fresh extract, these charts answer the operational questions — mix, magnitude, and timing — in seconds.